# 02 — Plot a FET Transfer Run

This notebook demonstrates how to:
1. Pick a published run from the catalog.
2. Load and parse the raw data via `labdata.store.load_run_data`.
3. Plot voltage vs. current (transfer curve).
4. Display the computed metrics from the review artifact.

**Access model:** read-only raw data via `labdata.store`; catalog queries via `labdata.catalog`.
Connects to the database as `labdata_nb` (read-only role).
`LABDATA_ROOT` environment variable sets the data root (default `/srv/labdata`).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import json

import labdata.catalog as catalog
import labdata.store as store

## 1. Pick a published run from the catalog

We query `v_runs` filtered to `state='published'` and pick the first result.
If your catalog is empty, run the `labdata.seed.seed_demo()` helper first (see
`shared/reproducibility_demo.ipynb` for an example).

In [ ]:
df_published = catalog.list_runs(state='published')

if df_published.empty:
    raise RuntimeError(
        "No published runs found. Seed demo data first:\n"
        "  from labdata.seed import seed_demo; seed_demo()"
    )

run_id = df_published.iloc[0]['id']
print(f"Selected run: {run_id}")
print(f"Sample:       {df_published.iloc[0].get('sample_id')}")
print(f"Device:       {df_published.iloc[0].get('device_id')}")
print(f"Type:         {df_published.iloc[0].get('measurement_type')}")

## 2. Load and parse the raw data

`store.load_run_data(run_id)` resolves the raw file path (via `catalog.get_run`),
reads the primary data file, and returns a `labdata.parsers.fet_transfer.ParseResult`
with `voltage` and `current` numpy arrays.

In [ ]:
result = store.load_run_data(run_id)

print(f"Parse ok:      {result.ok}")
print(f"Rows actual:   {result.rows_actual}")
if result.warnings:
    print(f"Warnings:      {result.warnings}")
if not result.ok:
    raise RuntimeError(f"Parse failed: {result.error}")

## 3. Plot the transfer curve

A FET transfer curve shows drain current (log scale) vs. gate voltage.
The on/off ratio is visible as the dynamic range of the current.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: log scale (shows on/off ratio clearly)
ax = axes[0]
ax.semilogy(result.voltage, np.abs(result.current), 'b-o', markersize=3, linewidth=1.5)
ax.set_xlabel('Gate Voltage (V)')
ax.set_ylabel('|Drain Current| (A)')
ax.set_title(f'FET Transfer Curve (log) — {run_id[:12]}...')
ax.grid(True, which='both', alpha=0.3)

# Right: linear scale
ax = axes[1]
ax.plot(result.voltage, result.current, 'r-o', markersize=3, linewidth=1.5)
ax.set_xlabel('Gate Voltage (V)')
ax.set_ylabel('Drain Current (A)')
ax.set_title(f'FET Transfer Curve (linear) — {run_id[:12]}...')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Review artifact metrics

`catalog.review_artifact(run_id)` returns the latest row from `v_review_artifacts`,
which contains pre-computed metrics stored when Pipeline A ran for this run.

In [ ]:
artifact = catalog.review_artifact(run_id)

if artifact is None:
    print("No review artifact found for this run.")
else:
    metrics = artifact.get('metrics_json', {})
    # metrics_json may be stored as a JSON string or a dict
    if isinstance(metrics, str):
        metrics = json.loads(metrics)

    print("Review artifact metrics:")
    print(f"  I_max:         {metrics.get('I_max', 'N/A'):.3e} A")
    print(f"  I_min:         {metrics.get('I_min', 'N/A'):.3e} A")
    print(f"  On/off ratio:  {metrics.get('on_off_ratio', 'N/A'):.2e}")
    vth = metrics.get('V_th')
    if vth is not None:
        print(f"  V_th estimate: {vth:.3f} V")
    else:
        print(f"  V_th estimate: N/A")

    if artifact.get('llm_summary'):
        print(f"\nLLM summary:\n  {artifact['llm_summary']}")

## 5. Raw file listing for this run

`catalog.run_files(run_id)` returns a DataFrame from `v_run_files` with the
name, size, and sha256 of each file in the raw archive.

In [ ]:
df_files = catalog.run_files(run_id)
print(f"Files in run {run_id[:12]}...:")
df_files[['name', 'bytes', 'sha256']]